# 📱 MultiVideo Downloader — Build APK en Colab

Compila el APK debug directo desde GitHub (`llxll4ever/MultiVideoDownloader`, rama `master`).

**Uso: Entorno de ejecución → Ejecutar todo** (o `Ctrl+F9`). Primera vez tarda ~6-9 min.

Al final descarga `MultiVideoDownloader-debug.zip` con 3 APKs:
- `app-arm64-v8a-debug.apk` → celulares modernos (S24, OnePlus 8 Pro)
- `app-armeabi-v7a-debug.apk` → celulares viejos 32-bit
- `app-universal-debug.apk` → compatible con todo (más pesado)

In [ ]:
# 1) Instalar JDK 17 + Android SDK 34 (~3 min)
import os
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!rm -rf /opt/android-sdk commandlinetools-linux-11076708_latest.zip
!mkdir -p /opt/android-sdk/cmdline-tools
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
!unzip -q commandlinetools-linux-11076708_latest.zip -d /opt/android-sdk/cmdline-tools
!mv /opt/android-sdk/cmdline-tools/cmdline-tools /opt/android-sdk/cmdline-tools/latest
os.environ["ANDROID_HOME"] = "/opt/android-sdk"
os.environ["PATH"] = f"/opt/android-sdk/cmdline-tools/latest/bin:{os.environ['PATH']}"
!yes | sdkmanager --licenses > /dev/null 2>&1
!sdkmanager "platforms;android-34" "build-tools;34.0.0" > /dev/null
!java -version 2>&1 | head -1 && sdkmanager --version | head -1

In [ ]:
# 2) Clonar repo y apuntar al SDK (~10 s)
%cd /content
!rm -rf MultiVideoDownloader
!git clone --depth 1 https://github.com/llxll4ever/MultiVideoDownloader.git
%cd MultiVideoDownloader
with open("local.properties", "w") as f:
    f.write("sdk.dir=/opt/android-sdk\n")
!chmod +x gradlew
!git log --oneline -1

In [ ]:
# 3) Compilar APK debug (~4-6 min)
!./gradlew assembleDebug --console=plain

In [ ]:
# 4) Empaquetar y descargar los APKs
import glob, shutil, os
from google.colab import files
apks = sorted(glob.glob("app/build/outputs/apk/**/*.apk", recursive=True))
assert apks, "No se generó ningún APK — revisa errores en la celda 3"
for a in apks:
    print(f"{os.path.getsize(a)/1e6:6.1f} MB  {a}")
!zip -q -j MultiVideoDownloader-debug.zip {" ".join(apks)}
files.download("MultiVideoDownloader-debug.zip")
print("✅ Listo: MultiVideoDownloader-debug.zip")